# CrisisLens · Custom CNN Training (Kaggle GPU)

從頭訓練自建 CNN 做 5 類災情分類，並跑 3 個 ablation 對照（no-BN / big-FC / shallow）。

## 執行前
1. Notebook options → Accelerator → GPU T4 x2（或 P100）
2. Internet → On
3. Add data → `mikolajbabula/disaster-images-dataset-cnn-model`
4. 確認下方 Config cell 的 `DATA_DIR`

## 輸出
- `custom_cnn.pth` — v1 final 權重（~1.5 MB）
- `custom_cnn_classes.json` — 類別對照
- `training_curves.png`、`ablation_comparison.png`、`confusion_matrix.png`

In [ ]:
import sys, torch
print(f"Python:     {sys.version.split()[0]}")
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.version.cuda}")
print(f"GPU 可用:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 名稱:   {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠️ 沒偵測到 GPU — 請從右側 Notebook options 啟用 GPU 後 Restart Session")

## 1. Config

In [ ]:
from pathlib import Path

DATA_DIR     = "/kaggle/input/disaster-images-dataset-cnn-model"
OUT_DIR      = Path("/kaggle/working")
WEIGHTS_PATH = OUT_DIR / "custom_cnn.pth"
MAPPING_PATH = OUT_DIR / "custom_cnn_classes.json"
CURVES_PATH  = OUT_DIR / "training_curves.png"
ABLATION_PATH= OUT_DIR / "ablation_comparison.png"
CM_PATH      = OUT_DIR / "confusion_matrix.png"

BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
EPOCHS        = 15
NUM_WORKERS   = 2
SEED          = 42
VAL_RATIO     = 0.2

FOLDER_TO_ZH = {
    "Damaged_Infrastructure": "地震或建築損壞",
    "Fire_Disaster":          "火災",
    "Land_Disaster":          "土石流或坍方",
    "Water_Disaster":         "淹水",
    "Non_Damage":             "其他或無明顯災害",
}

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

## 2. Imports

In [ ]:
import os, json, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(SEED)
np.random.seed(SEED)

## 3. 確認資料集結構

In [ ]:
def find_data_root(base: str) -> str:
    base_path = Path(base)
    if not base_path.exists():
        raise FileNotFoundError(f"{base} 不存在 — 請確認資料集已 attach 並更新 DATA_DIR")
    expected = set(FOLDER_TO_ZH.keys())

    def matches(p: Path) -> bool:
        if not p.is_dir():
            return False
        subs = {x.name for x in p.iterdir() if x.is_dir()}
        return expected.issubset(subs)

    if matches(base_path):
        return str(base_path)
    for sub in base_path.rglob("*"):
        if matches(sub):
            return str(sub)
    raise FileNotFoundError(
        f"在 {base} 下找不到包含 {expected} 五個資料夾的層級。"
    )

DATA_ROOT = find_data_root(DATA_DIR)
print(f"✅ 資料根目錄: {DATA_ROOT}\n")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
print("類別數量：")
total_imgs = 0
for folder, zh in FOLDER_TO_ZH.items():
    fp = Path(DATA_ROOT) / folder
    n = sum(1 for p in fp.rglob("*") if p.suffix.lower() in IMG_EXTS) if fp.exists() else 0
    total_imgs += n
    mark = "✅" if n > 0 else "❌"
    print(f"  {mark} {folder:25s} → {zh:12s}  {n:5d} 張")
print(f"\n總圖片數: {total_imgs}")